# Project Jarvis M40 — Kaggle GPU fine-tuning
Enable a **GPU accelerator** and Internet in Notebook options. Add the private M40 ZIP as notebook data, then run each cell once from top to bottom.

In [ ]:
from pathlib import Path
import subprocess
import sys

WORK = Path('/kaggle/working')
REPO = WORK / 'piper1-gpl'
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'cmake', 'ninja-build'], check=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'v1.4.2', '--depth', '1', 'https://github.com/OHF-Voice/piper1-gpl.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-build', 'onnxscript'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train]'], cwd=REPO, check=True)
subprocess.run(['bash', 'build_monotonic_align.sh'], cwd=REPO, check=True)
subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=REPO, check=True)
assert any((REPO / 'src/piper').glob('espeakbridge*.so')), 'Piper espeakbridge build missing'
print('M40 environment ready')

Before running this cell, use **Add Input** to attach the private M40 dataset. Kaggle extracts uploaded archives under `/kaggle/input`, so the cell discovers `DATASET_CARD.json` and copies the extracted dataset into the writable workspace.

In [ ]:
import json
import shutil
import wave

matches = list(Path('/kaggle/input').rglob('DATASET_CARD.json'))
assert len(matches) == 1, f'Expected one DATASET_CARD.json under /kaggle/input, found {len(matches)}'
source_dataset = matches[0].parent
dataset = WORK / 'm40-dataset'
shutil.rmtree(dataset, ignore_errors=True)
shutil.copytree(source_dataset, dataset)
card = json.loads((dataset / 'DATASET_CARD.json').read_text(encoding='utf-8'))
assert card['clip_count'] == 826
assert len(list((dataset / 'wav').glob('*.wav'))) == 826
assert (dataset / 'metadata.csv').exists()
source_metadata = (dataset / 'metadata.csv').read_text(encoding='utf-8').splitlines()
safe_metadata = dataset / 'metadata-kaggle-safe.csv'
safe_rows = []
safe_seconds = 0.0
for row in source_metadata:
    fields = row.split('|')
    if len(fields) < 2:
        continue
    audio_name, transcript = fields[0], fields[1]
    audio_path = dataset / 'wav' / (audio_name if audio_name.endswith('.wav') else f'{audio_name}.wav')
    with wave.open(str(audio_path), 'rb') as stream:
        duration = stream.getnframes() / stream.getframerate()
    if duration <= 6.0 and len(transcript) <= 120:
        safe_rows.append(row)
        safe_seconds += duration
assert len(safe_rows) >= 300, f'Only {len(safe_rows)} memory-safe clips remain'
safe_metadata.write_text('\n'.join(safe_rows) + '\n', encoding='utf-8')
print(f"Dataset verified: {card['clip_count']} clips, {card['duration_seconds']:.2f} seconds")
print(f'Memory-safe training subset: {len(safe_rows)} clips, {safe_seconds:.2f} seconds')

Prepare the trusted official checkpoint and train. This cell prints live Piper progress and writes checkpoints under `/content/jarvis-training`.

In [ ]:
import os
import urllib.request
import torch
import lightning.pytorch.cli as lightning_cli

BASE = WORK / 'en_GB_base.ckpt'
COMPAT = WORK / 'en_GB_base_compat.ckpt'
OUTPUT = WORK / 'jarvis-training'
CACHE = WORK / 'jarvis-cache'
OUTPUT.mkdir(exist_ok=True)
CACHE.mkdir(exist_ok=True)
if not BASE.exists():
    urllib.request.urlretrieve('https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_GB/northern_english_male/medium/epoch%3D9029-step%3D2261720.ckpt?download=true', BASE)
checkpoint = torch.load(BASE, weights_only=False, map_location='cpu')
checkpoint['hyper_parameters'] = {}
checkpoint['epoch'] = 0
checkpoint['global_step'] = 0
progress_keys = {'ready', 'started', 'processed', 'completed'}
def reset_progress(value):
    if isinstance(value, dict):
        for key, child in list(value.items()):
            if key in progress_keys and isinstance(child, int):
                value[key] = 0
            else:
                reset_progress(child)
    elif isinstance(value, list):
        for child in value:
            reset_progress(child)
reset_progress(checkpoint.get('loops', {}))
torch.save(checkpoint, COMPAT)

cli_path = Path(lightning_cli.__file__)
cli_text = cli_path.read_text(encoding='utf-8')
old_load = 'torch.load(ckpt_path, weights_only=True, map_location="cpu")'
trusted_load = 'torch.load(ckpt_path, weights_only=False, map_location="cpu")'
assert old_load in cli_text or trusted_load in cli_text
if old_load in cli_text:
    cli_path.write_text(cli_text.replace(old_load, trusted_load, 1), encoding='utf-8')

command = [
    sys.executable, '-u', '-m', 'piper.train', 'fit',
    '--data.voice_name', 'jarvis',
    '--data.csv_path', str(safe_metadata),
    '--data.audio_dir', str(dataset / 'wav'),
    '--model.sample_rate', '22050',
    '--data.espeak_voice', 'en',
    '--data.cache_dir', str(CACHE),
    '--data.config_path', str(OUTPUT / 'en_GB-jarvis-medium.onnx.json'),
    '--data.batch_size', '1',
    '--ckpt_path', str(COMPAT),
    '--trainer.accelerator', 'gpu',
    '--trainer.devices', '1',
    '--trainer.precision', '16-mixed',
    '--trainer.max_epochs', '60',
    '--trainer.default_root_dir', str(OUTPUT),
    '--weights_only', 'true',
]
environment = os.environ.copy()
environment['PYTHONPATH'] = str(REPO / 'src')
environment['PYTHONUNBUFFERED'] = '1'
environment['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'Starting M40 Piper fine-tune (60 epochs, {len(safe_rows)} memory-safe clips)')
subprocess.run(command, cwd=REPO, env=environment, check=True)

Export the newest checkpoint and download the private ONNX package.

In [ ]:
import zipfile

checkpoints = sorted(OUTPUT.rglob('*.ckpt'), key=lambda path: path.stat().st_mtime)
assert checkpoints, 'Training produced no checkpoint'
latest = checkpoints[-1]
model = OUTPUT / 'en_GB-jarvis-medium.onnx'
config = OUTPUT / 'en_GB-jarvis-medium.onnx.json'
export_source = REPO / 'src/piper/train/export_onnx.py'
export_text = export_source.read_text(encoding='utf-8')
legacy_axes = '        dynamic_axes={'
legacy_export = '        dynamo=False,\n        dynamic_axes={'
assert legacy_axes in export_text or legacy_export in export_text
if legacy_export not in export_text:
    export_source.write_text(export_text.replace(legacy_axes, legacy_export, 1), encoding='utf-8')
subprocess.run([sys.executable, '-m', 'piper.train.export_onnx', '--checkpoint', str(latest), '--output-file', str(model)], cwd=REPO, env=environment, check=True)
package = WORK / 'jarvis-piper-m40.zip'
with zipfile.ZipFile(package, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(model, model.name)
    bundle.write(config, config.name)
print('Export ready:', package)
print('Download it from the Kaggle Output/Files panel after the cell completes.')